# TP - Qualité des données : valeurs manquantes
## Spaceship Titanic

**Durée indicative : 1 h 30**

### Objectifs

À partir d'un jeu de données volontairement détérioré, l'objectif est de :

1. quantifier et visualiser les valeurs manquantes ;
2. étudier leurs motifs ;
3. rechercher des indices permettant de suspecter un mécanisme **MCAR**, **MAR** ou **MNAR** ;
4. choisir des stratégies de traitement simples et adaptées ;
5. vérifier qu'une imputation ne signifie pas nécessairement que la qualité des données est restaurée.


## 0. Contexte

Le vaisseau *Spaceship Titanic* transporte plusieurs milliers de passagers.

À la suite de différents problèmes de collecte, certaines informations sont absentes.  
Trois variables sont concernées :

- `HomePlanet` ;
- `CryoSleep` ;
- `RoomService`.

Votre mission consiste à **diagnostiquer la structure des valeurs manquantes avant de les traiter**.

Les autres colonnes utiles sont notamment :

- `Age` : âge du passager ;
- `Deck` / `CabinSide` : localisation de la cabine ;
- `VIP` : statut VIP ;
- `FoodCourt`, `Spa` : dépenses à bord ;

## 1. Chargement et première exploration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/spaceship_titanic_missing.csv")

df = pd.read_csv(DATA_PATH)

print("Dimensions :", df.shape)
display(df.head())

### Exercice 1 — Comprendre le dataset

1. Affichez les types des variables.
2. Vérifiez l'unicité de `PassengerId`.
3. Affichez quelques statistiques descriptives sur les variables numériques.
4. Identifiez les variables qui contiennent des valeurs manquantes.

In [ ]:
# Votre réponse

## 2. Quantifier les valeurs manquantes

Avant toute correction, on mesure l'ampleur du problème.

### Exercice 2

Construisez un tableau contenant, pour chaque variable :

- le nombre de valeurs manquantes ;
- le pourcentage de valeurs manquantes.

Ne conservez que les variables concernées et triez-les par pourcentage décroissant.

In [ ]:
# Votre réponse

### Exercice 3 — Combien de lignes sont réellement concernées ?

Calculez :

- le nombre de lignes incomplètes ;
- leur proportion dans le dataset ;
- le nombre total de valeurs manquantes ;
- le nombre de valeurs manquantes par ligne.

In [ ]:
# Votre réponse

## 3. Visualiser les valeurs manquantes

### Exercice 4

Représentez le pourcentage de valeurs manquantes par variable avec un diagramme en barres.

Le but n'est pas de produire une visualisation complexe : on veut simplement voir immédiatement quelles variables sont concernées.

In [ ]:
# Votre réponse

## 4. Étudier les motifs de valeurs manquantes

Pour chaque variable incomplète, on peut créer un indicateur binaire :

- `1` : valeur manquante ;
- `0` : valeur présente.

### Exercice 5

Créez trois indicateurs :

- `HomePlanet_missing`
- `CryoSleep_missing`
- `RoomService_missing`

Puis affichez leur matrice de corrélation.

**Question :** les absences semblent-elles fortement liées entre elles ?

In [ ]:
# Votre réponse

## 5. Diagnostic du mécanisme de `HomePlanet`

### Exercice 6

Comparez les passagers pour lesquels `HomePlanet` est renseignée à ceux pour lesquels elle est absente.

Commencez simplement par comparer l'âge :

- effectif ;
- moyenne ;
- médiane.

Vous pouvez également regarder une autre variable numérique si vous le souhaitez.

**Question :** observez-vous une différence importante entre les deux groupes ?

In [ ]:
# Votre réponse

## 6. Diagnostic du mécanisme de `CryoSleep`

### Exercice 7

On cherche maintenant à savoir si l'absence de `CryoSleep` peut être liée à une variable observée.

Comparez l'âge des passagers selon que `CryoSleep` est :

- renseignée ;
- manquante.

Calculez au minimum la moyenne et la médiane, puis représentez les deux distributions avec un boxplot.

**Question :** que remarquez-vous ?

In [ ]:
# Votre réponse

## 7. Et `RoomService` ?

### Exercice 8

Étudiez `RoomService` :

1. affichez sa distribution parmi les valeurs observées ;
2. comparez quelques variables disponibles entre les lignes où `RoomService` est renseigné et celles où il est manquant ;
3. essayez de déterminer si le mécanisme est MCAR, MAR ou MNAR.

**Question importante :** peut-on savoir, à partir du dataset incomplet uniquement, si l'absence de `RoomService` dépend de la valeur de `RoomService` qui a disparu ?

In [ ]:
# Votre réponse

## 8. Choisir une stratégie de traitement

On dispose maintenant de trois situations différentes :

| Variable | Mécanisme suspecté | Taux |
|---|---|---:|
| `HomePlanet` | MCAR | ≈ 5 % |
| `CryoSleep` | MAR lié à `Age` | ≈ 8 % |
| `RoomService` | MNAR | ≈ 7 % |

L'objectif n'est pas de trouver une méthode universelle, mais de choisir un traitement cohérent avec chaque situation.

### Exercice 9 — `HomePlanet`

Le taux de valeurs manquantes est faible et le mécanisme est compatible avec MCAR.

1. Combien de lignes seraient perdues si l'on supprimait toutes les lignes où `HomePlanet` est manquant ?
2. Proposez une stratégie simple.
3. Pour la suite du TP, imputez `HomePlanet` par sa modalité la plus fréquente.

In [ ]:
# Votre réponse

### Exercice 10 — `CryoSleep`

Le mécanisme est compatible avec MAR et semble dépendre de `Age`.

On va utiliser une méthode simple d'**imputation par modèle** :

1. entraînez une régression logistique sur les lignes où `CryoSleep` est connu ;
2. utilisez `Age`, `FoodCourt` et `Spa` comme variables explicatives ;
3. prédisez les valeurs manquantes de `CryoSleep`.

Aucune optimisation du modèle n'est demandée.


In [ ]:
# Votre réponse

### Exercice 11 — `RoomService`

Pour `RoomService`, le mécanisme est suspecté MNAR.

Dans ce cas, l'absence peut elle-même être informative.

1. créez un indicateur `RoomService_missing` ;
2. imputez ensuite les valeurs manquantes avec la médiane observée ;
3. conservez bien l'indicateur dans le dataset.

Pourquoi l'indicateur est-il utile ?

In [ ]:
# Votre réponse

## 9. Contrôle final du dataset traité

### Exercice 12

1. vérifiez qu'il ne reste plus de valeurs manquantes sur les trois variables étudiées ;
2. comparez rapidement le dataset avant et après traitement ;
3. réfléchissez à ce qui a réellement été « corrigé » et à ce qui reste incertain.

In [ ]:
# Votre réponse

## 10. Synthèse

À retenir :

1. **ne pas commencer par imputer** ;
2. quantifier et visualiser les absences ;
3. créer des indicateurs de manque et rechercher des relations ;
4. comparer les observations complètes et incomplètes ;
5. raisonner en termes de **MCAR / MAR / MNAR** ;
6. choisir un traitement adapté au mécanisme et à l'usage ;
7. vérifier autant que possible l'effet du traitement sur les données.
